In [ ]:

# --- SETUP & CONFIG ---

from __future__ import annotations  # allow forward type hints in older Python
from dataclasses import dataclass, asdict  # structured records and export helpers
from pathlib import Path  # path manipulations
from typing import List, Dict, Tuple, Optional  # type hints for clarity
import time  # polite delays between requests
import hashlib  # stable cache filenames from URLs
import json  # save metadata and structured records
import re  # text cleanup and field parsing
import urllib.robotparser as robotparser  # robots.txt compliance
import urllib.parse as urlparse  # resolve relative links

import os  # environment settings
from openai import OpenAI  # Hugging Face router-compatible client
from huggingface_hub import InferenceClient  # HF inference helper
import requests  # HTTP client
from bs4 import BeautifulSoup  # HTML parsing (pip install beautifulsoup4)
import numpy as np  # numerical arrays for embeddings
import faiss  # vector search (pip install faiss-cpu)

# ----------------------- USER-EDITABLE CONFIG -----------------------

# Root index page that lists all course links (replace with your real URL)
CATALOG_INDEX_URL = "https://www.uaeu.ac.ae/en/catalog/undergraduate/programs/bachelor-of-science-in-computer-science.shtml"  # placeholder index URL

# Auto-discovery/extraction heuristics (see below) follow links and parse course fields dynamically.

# Local folders for artifacts

DATA_ROOT   = Path("./web_corpus")     # stores raw HTML cache and extracted JSON

INDEX_ROOT  = Path("./rag_index")      # stores FAISS vectors + metadata

DATA_ROOT.mkdir(parents=True, exist_ok=True)

INDEX_ROOT.mkdir(parents=True, exist_ok=True)

# Load variables from a local .env (optional convenience)
ENV_FILE = Path(".env")
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        if not line or line.lstrip().startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

# Polite crawling parameters
USER_AGENT       = "StudentAssistantBot/1.0 (+https://yourdomain.example; contact@example.edu)"
REQUEST_TIMEOUT  = 30  # seconds per HTTP request / max seconds the client waits for a response before aborting the request
CRAWL_DELAY_SEC  = 1.0  # polite delay between HTTP requests
USE_CACHE        = True  # set False to bypass cache and re-fetch
FORCE_REFRESH    = False # set True to ignore cache and re-fetch everything

# Hugging Face Inference API configuration
HF_API_TOKEN       = os.environ.get("HF_API_TOKEN")  # set this env var before running
HF_ROUTER_BASE     = os.environ.get("HF_ROUTER_BASE", "https://router.huggingface.co/v1")
HF_EMBED_MODEL     = "BAAI/bge-small-en"                    # embedding model (HF inference)
GEN_TEMPERATURE    = 0.2                    # low temperature for precision
GEN_TOP_P          = 0.95                   # conservative nucleus sampling
GEN_MAX_TOKENS     = 600                    # cap to keep latency reasonable
TOP_K_RETRIEVAL    = 5                      # number of passages to retrieve


In [ ]:
# --- HTTP + ROBOTS + CACHE ---

def cache_path_for(url: str) -> Path:
    """Return a stable cache file path derived from the URL."""
    # Hash the URL so the filename is filesystem-safe and unique
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()  # compute SHA-1 hex
    return DATA_ROOT / f"{h}.html"  # store as .html for convenience


def load_robots_parser(root_url: str) -> robotparser.RobotFileParser:
    """Load and parse robots.txt for the site hosting root_url."""
    # Compute robots.txt URL from the site origin
    parts = urlparse.urlparse(root_url)  # split URL into components
    robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"  # assemble robots.txt URL
    rp = robotparser.RobotFileParser()  # create parser instance
    try:
        rp.set_url(robots_url)  # set robots URL
        rp.read()  # fetch and parse robots.txt
    except Exception:
        # If robots fetch fails, we err on permissive interpretation but still crawl gently
        pass  # ignore errors; rp will allow by default
    return rp  # return parser


# not used at all
def polite_fetch(url: str,
                 rp: Optional[robotparser.RobotFileParser],
                 force_refresh: bool = FORCE_REFRESH,
                 use_cache: bool = USE_CACHE) -> str:
    """Fetch a URL with robots.txt check, delay, and local file cache."""
    # If caching is enabled and not forcing refresh, try to load from disk
    cpath = cache_path_for(url)  # compute cache path
    if use_cache and not force_refresh and cpath.exists():  # check if cached
        return cpath.read_text(encoding="utf-8", errors="ignore")  # return cached HTML
    
    # If robots.txt is available, verify that fetching is allowed
    allowed = True if rp is None else rp.can_fetch(USER_AGENT, url)  # check rule
    if not allowed:  # disallow if robots forbids
        raise PermissionError(f"robots.txt disallows fetching: {url}")  # fail fast
    
    # Polite delay before real network call
    time.sleep(CRAWL_DELAY_SEC)  # sleep to reduce load on origin
    
    # Perform the HTTP GET with headers and timeout
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=REQUEST_TIMEOUT)  # send request
    resp.raise_for_status()  # raise for non-200 status codes
    
    # Decode text (requests handles charset heuristics)
    html = resp.text  # get HTML content as text
    
    # Save to cache if enabled
    if use_cache:  # only cache when enabled
        cpath.write_text(html, encoding="utf-8")  # persist raw HTML
    
    return html  # return HTML string


In [ ]:
# --- AUTO DISCOVERY: FIND UAEU COURSE PAGES FROM THE PROGRAM ROOT ---

from urllib.parse import urljoin, urlparse  # URL utilities
import requests                             # HTTP client
import re                                   # regex helpers

ROOT_URL = CATALOG_INDEX_URL                # UAEU CS program page
ALLOWED_HOST = "uaeu.ac.ae"                 # keep links on this host
COURSE_PATH_RE = re.compile(r"/catalog/courses/course_\d+\.shtml", re.I)
TARGET_SECTIONS = {
    "major requirements (req. ch: 40)",
    "major electives (req. ch: 12)",
    "college of information technology (req. ch: 41)",
}
HEADINGS_TO_STOP = {"h2", "h3", "h4"}


def fetch(url: str) -> str:
    """Fetch a URL with a clear User-Agent and return HTML text."""
    response = requests.get(
        url,
        headers={"User-Agent": "StudentAssistantBot/1.0 (+contact@example.edu)"},
        timeout=30,
    )
    response.raise_for_status()
    return response.text


def normalize_label(text: str) -> str:
    """Lowercase label text and collapse whitespace for comparison."""
    return " ".join(text.lower().split())


def discover_course_urls(root_url: str) -> list[str]:
    """Return absolute URLs for College Requirement/Major Requirement/Elective course pages."""
    html = fetch(root_url)
    soup = BeautifulSoup(html, "html.parser")
    urls: list[str] = []

    for heading in soup.find_all(["h3", "h4", "h5"]):
        label = normalize_label(heading.get_text(" ", strip=True))
        if label not in TARGET_SECTIONS:
            continue

        node = heading
        while node:
            node = node.find_next_sibling()
            if node is None:
                break
            name = getattr(node, "name", None)
            if name and name.lower() in HEADINGS_TO_STOP:
                break
            if name is None:
                continue

            for anchor in node.find_all("a", href=True):
                abs_url = urljoin(root_url, anchor["href"])
                parsed = urlparse(abs_url)
                if parsed.netloc.lower().endswith(ALLOWED_HOST) and COURSE_PATH_RE.search(parsed.path):
                    urls.append(abs_url)

    seen: set[str] = set()
    unique: list[str] = []
    for url in urls:
        if url not in seen:
            unique.append(url)
            seen.add(url)
    return unique


course_urls = discover_course_urls(ROOT_URL)
print(f"Discovered {len(course_urls)} course pages across target sections.")
print(*course_urls[:5], sep="\n")



Discovered 41 course pages across target sections.
https://www.uaeu.ac.ae/en/catalog/courses/course_2968.shtml?id=CSBP119
https://www.uaeu.ac.ae/en/catalog/courses/course_2943.shtml?id=MATH105
https://www.uaeu.ac.ae/en/catalog/courses/course_2957.shtml?id=PHYS105
https://www.uaeu.ac.ae/en/catalog/courses/course_2948.shtml?id=CENG202
https://www.uaeu.ac.ae/en/catalog/courses/course_2948.shtml?id=CENG205


In [ ]:
# --- AUTO EXTRACTION: ROBUST HEURISTICS FOR COURSE FIELDS ---

from dataclasses import dataclass                        # structured records
from urllib.parse import parse_qs, urlparse              # query parsing helpers
from bs4 import BeautifulSoup                            # DOM parsing
import re                                                # regex

RE_CODE = re.compile(r"[A-Z]{2,5}\s?-?\s?\d{3}")
RE_CREDITS_TRAILING = re.compile(r"(\d+)\s*(credit(?:s)?|credit hours)", re.I)
RE_CREDITS_LEADING = re.compile(r"credit(?:s)?(?: hours)?\s*:?\s*(\d+)", re.I)
RE_PREREQ_LINE = re.compile(r"pre[\-\s]?requisite", re.I)
RE_COREQ_LINE = re.compile(r"co[\-\s]?requisite", re.I)


@dataclass
class Course:
    url: str
    code: str
    title: str
    credits: str
    prerequisites: str
    description: str
    full_text: str


def text_clean(s: str) -> str:
    """Normalize whitespace for consistent downstream use."""
    s = s.replace(" ", " ")
    s = s.replace("\r", "\n")
    s = re.sub(r"[ 	]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def course_content_root(soup: BeautifulSoup, url: str):
    """Return the div whose id matches the course id=... query parameter."""
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    raw_id = ""
    for key in ("id", "course"):
        val = query.get(key)
        if val:
            raw_id = val[0].strip()
            break
    if raw_id:
        for candidate in (raw_id, raw_id.upper(), raw_id.lower()):
            if not candidate:
                continue
            found = soup.find(id=candidate)
            if found:
                return found, raw_id
    return soup, raw_id


def iter_labeled_segments(root) -> list[tuple[str, str]]:
    """Yield (heading_label, text) pairs in document order within the course div."""
    segments: list[tuple[str, str]] = []
    current_label: str | None = None
    for child in root.children:
        if getattr(child, "name", None):
            text = child.get_text(" ", strip=True)
            clean = text_clean(text)
            if not clean:
                continue
            name = child.name.lower()
            if name in {"h2", "h3", "h4", "h5", "strong"}:
                current_label = normalize_label(clean)
            segments.append((current_label, clean))
        else:
            clean = text_clean(str(child))
            if clean:
                segments.append((current_label, clean))
    return segments


def find_section_text(segments, predicate) -> str:
    """Return text belonging to the first heading whose label/body satisfies predicate."""
    for idx, (label, text) in enumerate(segments):
        target_label = label or ""
        if predicate(target_label) or predicate(text):
            if predicate(text) and len(text.split()) > 4:
                return text
            collected: list[str] = []
            base = label
            j = idx + 1
            while j < len(segments):
                next_label, next_text = segments[j]
                if next_label != base:
                    break
                collected.append(next_text)
                j += 1
            if collected:
                return " ".join(collected)
            return text
    return ""


def detect_credits(segments, text_blob: str) -> str:
    """Find the credit hours string either in segmented headings or the raw blob."""
    def from_text(value: str) -> str:
        match = RE_CREDITS_TRAILING.search(value)
        if match:
            return f"{match.group(1)} credits"
        match = RE_CREDITS_LEADING.search(value)
        if match:
            return f"{match.group(1)} credits"
        return ""

    for _, text in segments:
        found = from_text(text)
        if found:
            return found
    return from_text(text_blob)


def extract_code_from_url(url: str, raw_id: str) -> str:
    """Attempt to get the course code from the URL query, falling back to the raw id."""
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    candidate = (query.get("id") or query.get("course") or [])
    if candidate:
        raw = candidate[0].strip().upper()
        match = re.match(r"([A-Z]{2,5})\s?-?\s?(\d{3})", raw)
        if match:
            return f"{match.group(1)} {match.group(2)}"
        return raw
    return raw_id.upper() if raw_id else ""


def extract_course_fields(url: str) -> Course:
    """Fetch a course page and fill Course fields using heuristics."""
    html = fetch(url)
    soup = BeautifulSoup(html, "html.parser")
    content_root, raw_id = course_content_root(soup, url)

    h1 = content_root.find("h1") or soup.find("h1")
    title = text_clean(h1.get_text(" ", strip=True)) if h1 else ""

    paragraph_nodes = content_root.find_all(["p", "li"])
    paragraphs = [text_clean(p.get_text(" ", strip=True)) for p in paragraph_nodes if p.get_text(strip=True)]

    segments = iter_labeled_segments(content_root)
    text_blob = text_clean(content_root.get_text("\n", strip=True))

    code = extract_code_from_url(url, raw_id)
    if not code:
        for text in paragraphs:
            match = RE_CODE.search(text)
            if match:
                code = re.sub(r"[\s-]+", " ", match.group(0).upper()).strip()
                break

    credits = detect_credits(segments, text_blob)

    prereq = find_section_text(segments, lambda value: bool(RE_PREREQ_LINE.search(value or "")))
    if not prereq:
        coreq = find_section_text(segments, lambda value: bool(RE_COREQ_LINE.search(value or "")))
        if coreq:
            prereq = f"Corequisite: {coreq}"

    desc_candidates = [t for t in paragraphs if len(t.split()) > 8 and not RE_PREREQ_LINE.search(t)]
    description = max(desc_candidates, key=lambda s: len(s), default="")

    full_parts = [code, title, credits, prereq, description]
    full_text = text_clean("\n\n".join([part for part in full_parts if part]))

    return Course(url=url, code=code, title=title, credits=credits,
                  prerequisites=prereq, description=description, full_text=full_text)


CourseRecord = Course  # backward compatibility for downstream chunking logic

sample = course_urls[:5]
records = [extract_course_fields(u) for u in sample]
for record in records:
    print(f"- {record.code} | {record.title} | {record.credits}")
    if record.prerequisites:
        print("  Prereq:", record.prerequisites)



- CSBP 119 | Algorithms and Problem Solving (CSBP119) | 3 credits
- MATH 105 | Calculus I (MATH105) | 3 credits
- PHYS 105 | General Physics I (PHYS105) | 3 credits
- CENG 202 | Discrete Mathematics (CENG202) | 3 credits
  Prereq: MATH105 with a minimum grade D
- CENG 205 | Digital Design & Computer Organization (CENG205) | 3 credits


In [206]:
# --- CHUNKING: BUILD CITES & METADATA ---

@dataclass
class Chunk:
    chunk_id: str
    url: str
    code: str
    title: str
    section: str
    text: str


def normalize_course_code(code: str) -> str:
    """Return a normalized 'ABCD 123' variant for matching."""
    if not code:
        return ""
    code = code.upper().strip()
    match = re.match(r"([A-Z]{2,5})\s*-?\s*(\d{3})", code)
    if match:
        return f"{match.group(1)} {match.group(2)}"
    return code


def course_code_aliases(code: str) -> List[str]:
    """Return compact/dashed aliases to stuff into each chunk."""
    norm = normalize_course_code(code)
    if not norm:
        return []
    compact = norm.replace(" ", "")
    dashed = norm.replace(" ", "-")
    return sorted({norm, compact, dashed})


def _chunk_text(course: CourseRecord, section: str, body: str) -> str:
    """Prefix body text with course code/title/section for better retrieval signals."""
    parts = [part for part in (course.code, course.title) if part]
    header = " | ".join(parts)
    if section:
        header = " - ".join([value for value in (header, section.title()) if value])
    alias_line = ""
    if course.code:
        aliases = course_code_aliases(course.code)
        if aliases:
            alias_line = "Aliases: " + " | ".join(aliases)
    text_parts = [value for value in (header, body, alias_line) if value]
    return "\n".join(text_parts)


def make_chunks(course: CourseRecord,
                max_chars: int = 1100,
                overlap: int = 120) -> List[Chunk]:
    """Create citable chunks from a CourseRecord with logical sections and fallback full text."""
    base = (course.code or course.title or "course").replace(" ", "_")

    chunks: List[Chunk] = []
    if course.description:
        text = _chunk_text(course, "description", course.description)
        chunks.append(Chunk(chunk_id=f"{base}::description", url=course.url,
                            code=course.code, title=course.title or course.code,
                            section="description", text=text))
    if course.prerequisites:
        text = _chunk_text(course, "prerequisites", course.prerequisites)
        chunks.append(Chunk(chunk_id=f"{base}::prerequisites", url=course.url,
                            code=course.code, title=course.title or course.code,
                            section="prerequisites", text=text))
    if course.credits:
        text = _chunk_text(course, "credits", course.credits)
        chunks.append(Chunk(chunk_id=f"{base}::credits", url=course.url,
                            code=course.code, title=course.title or course.code,
                            section="credits", text=text))

    if not chunks and course.full_text:
        text = course.full_text
        start = 0
        count = 0
        while start < len(text):
            end = min(start + max_chars, len(text))
            piece = text[start:end]
            cid = f"{base}::full_{count:02d}"
            chunks.append(Chunk(chunk_id=cid, url=course.url, code=course.code,
                                title=course.title or course.code, section="full", text=piece))
            if end >= len(text):
                break
            start = end - overlap
            count += 1

    return chunks


In [207]:

# --- HUGGING FACE CLIENTS, INDEXING, PERSISTENCE ---

_HF_CLIENT: Optional[OpenAI] = None
_HF_EMBED_CLIENT: Optional[InferenceClient] = None


def get_hf_client() -> OpenAI:
    """Lazily construct an OpenAI client pointed at the Hugging Face router for chat."""
    if not HF_API_TOKEN:
        raise RuntimeError(f"Set HF_API_TOKEN in your environment or in {ENV_FILE} before running the notebook.")
    global _HF_CLIENT
    if _HF_CLIENT is None:
        _HF_CLIENT = OpenAI(api_key=HF_API_TOKEN, base_url=HF_ROUTER_BASE)
    return _HF_CLIENT


def get_embed_client() -> InferenceClient:
    """Create or reuse the HF Inference client for embeddings."""
    if not HF_API_TOKEN:
        raise RuntimeError(f"Set HF_API_TOKEN in your environment or in {ENV_FILE} before running the notebook.")
    global _HF_EMBED_CLIENT
    if _HF_EMBED_CLIENT is None:
        _HF_EMBED_CLIENT = InferenceClient(provider="hf-inference", api_key=HF_API_TOKEN)
    return _HF_EMBED_CLIENT


def hf_embed(texts: List[str], mode: str = "document") -> np.ndarray:
    """Call the HF feature-extraction endpoint with BGE prefixes."""
    client = get_embed_client()
    prefix = "query: " if mode == "query" else "passage: "
    prefixed = [prefix + text.replace("\n", " ") for text in texts]
    data = client.feature_extraction(prefixed, model=HF_EMBED_MODEL)
    if data is None:
        raise RuntimeError("Received no embeddings from Hugging Face inference API.")
    if isinstance(data, np.ndarray):
        return data.astype("float32")
    if isinstance(data, list):
        if len(data) == 0:
            raise RuntimeError("Received empty embedding list from Hugging Face inference API.")
        if isinstance(data[0], list):
            vectors = data
        else:
            vectors = [data]
        return np.array(vectors, dtype="float32")
    raise RuntimeError(f"Unexpected embedding response type: {type(data)}")
def normalize_rows(v: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v / norms


def build_faiss_index(chunks: List[Chunk]) -> Tuple[faiss.IndexFlatIP, List[Chunk]]:
    texts = [c.text for c in chunks]
    embs = hf_embed(texts, mode="document")
    embs = normalize_rows(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, chunks


def persist_index(index: faiss.Index, chunks: List[Chunk]) -> None:
    faiss_path = INDEX_ROOT / "vectors.faiss"
    meta_path = INDEX_ROOT / "metadata.jsonl"
    faiss.write_index(index, str(faiss_path))
    with meta_path.open("w", encoding="utf-8") as f:
        for row_id, ch in enumerate(chunks):
            rec = {**asdict(ch), "row_id": row_id}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_index() -> Tuple[faiss.Index, List[Chunk]]:
    faiss_path = INDEX_ROOT / "vectors.faiss"
    meta_path = INDEX_ROOT / "metadata.jsonl"
    index = faiss.read_index(str(faiss_path))
    chunks: List[Chunk] = []
    with meta_path.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            chunks.append(Chunk(
                chunk_id=rec["chunk_id"], url=rec["url"], code=rec["code"],
                title=rec["title"], section=rec["section"], text=rec["text"]
            ))
    return index, chunks


In [211]:
# --- RETRIEVAL + HUGGING FACE GENERATION ---

CODE_PATTERN = re.compile(r"\b([A-Z]{2,5})[-\s]?(\d{3})\b")


def detect_course_codes(text: str) -> List[str]:
    """Extract normalized course codes from free-form text."""
    found = []
    for match in CODE_PATTERN.finditer(text.upper()):
        found.append(f"{match.group(1)} {match.group(2)}")
    seen = set()
    ordered: List[str] = []
    for code in found:
        if code not in seen:
            ordered.append(code)
            seen.add(code)
    return ordered


def retrieve(query: str, index: faiss.Index, chunks: List[Chunk], top_k: int = TOP_K_RETRIEVAL) -> List[Tuple[Chunk, float]]:
    query_codes = detect_course_codes(query)
    code_set = set(query_codes)
    code_rank = {code: rank for rank, code in enumerate(query_codes)}
    exact_hits: List[Tuple[int, int, Chunk]] = []  # (rank, idx, chunk)
    seen_indices = set()
    if code_set:
        for idx, ch in enumerate(chunks):
            normalized = normalize_course_code(ch.code or "")
            if normalized and normalized in code_set:
                rank = code_rank.get(normalized, len(code_rank))
                exact_hits.append((rank, idx, ch))
                seen_indices.add(idx)
    augmented_query = query.strip()
    if code_set:
        aliases = ", ".join(sorted(code_set))
        augmented_query = f"{augmented_query} (course codes: {aliases})"
    qvec = hf_embed([augmented_query], mode="query")
    qvec = normalize_rows(qvec)
    scores, ids = index.search(qvec, top_k)
    results: List[Tuple[Chunk, float]] = []
    for _, idx, ch in sorted(exact_hits, key=lambda item: (item[0], item[1])):
        results.append((ch, 1.1))
    for score, idx in zip(scores[0].tolist(), ids[0].tolist()):
        if idx == -1 or idx in seen_indices:
            continue
        seen_indices.add(idx)
        results.append((chunks[idx], float(score)))
        if len(results) >= top_k:
            break
    return results[:top_k]

SYSTEM_PROMPT = (
    "You are a precise Student Assistant for the United Arab Emirates University (UAEU) Computer Science program. "
    "Answer ONLY using the provided UAEU sources (domain must be uaeu.ac.ae). "
    "If the sources are insufficient, say: 'I do not have sufficient UAEU catalog information to answer this question.' "
    "Do NOT mention or recommend any non-UAEU institutions or contacts. "
    "Use formal language, correct punctuation, and cite like [code • section • chunk_id] for every factual claim."
)



def format_sources(evidence: List[Tuple[Chunk, float]]) -> str:
    blocks = []
    for rank, (ch, score) in enumerate(evidence, start=1):
        header = f"[{rank}] {ch.code or ch.title} • {ch.section} • {ch.chunk_id} • {ch.url} • sim={score:.3f}"
        blocks.append(header + "\n" + ch.text)
    return "\n\n---\n\n".join(blocks) if blocks else "(no sources)"


def hf_chat(system_prompt: str, user_prompt: str) -> str:
    client = get_hf_client()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = client.chat.completions.create(
        model=HF_CHAT_MODEL,
        messages=messages,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        max_tokens=GEN_MAX_TOKENS,
    )
    choice = response.choices[0]
    return (choice.message.content or "").strip()


def answer_query(query: str, index: faiss.Index, chunks: List[Chunk], top_k: int = TOP_K_RETRIEVAL) -> str:
    evidence = retrieve(query, index, chunks, top_k=top_k)
    context = format_sources(evidence)
    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Sources:\n{context}\n\n"
        "Instructions:\n"
        "- Answer using ONLY the information in the Sources above.\n"
        "- Provide inline citations for every factual statement using the pattern [code • section • chunk_id].\n"
        "- If the question asks for information that the Sources do not state, say so clearly and suggest the official contact.\n"
        "Never say you that you do not have additional or no information unless you are explicitly asked for such type of information.\n"
        "- If multiple sources conflict, prefer registrar/official catalog pages and note the discrepancy."
    )
    return hf_chat(SYSTEM_PROMPT, user_prompt)


In [ ]:

# --- PIPELINE ORCHESTRATION ---

def build_corpus_from_catalog(index_url: str) -> List[Chunk]:
    """Discover, harvest, extract, and chunk all courses from the catalog index."""
    course_urls = discover_course_urls(index_url)  # auto-discover UAEU course pages
    records: List[CourseRecord] = []  # accumulator
    for i, url in enumerate(course_urls, start=1):
        try:
            rec = extract_course_fields(url)  # heuristic extraction
            records.append(rec)
        except Exception as e:
            print(f"[WARN] Skipped {url}: {e}")
    all_chunks: List[Chunk] = []
    for rec in records:
        all_chunks.extend(make_chunks(rec))
    return all_chunks


# ---- BUILD OR LOAD INDEX ----
REBUILD = True  # set to False after first run to reuse saved index

if REBUILD:
    # Build chunks from the live catalog (index page → course pages)
    chunks = build_corpus_from_catalog(CATALOG_INDEX_URL)  # produce chunks
    # Construct FAISS index from chunk embeddings
    index, chunks = build_faiss_index(chunks)  # build vectors + index
    # Persist vectors and metadata for quick reuse
    persist_index(index, chunks)  # save artifacts
else:
    # Load previously persisted index and metadata
    index, chunks = load_index()  # restore artifacts


In [213]:
# ---- TEST A QUERY ----
sample_question = "Can you tell me about the course CSBP 119 offered by UAEU?"  # edit to match your catalog
response = answer_query(sample_question, index, chunks, top_k=TOP_K_RETRIEVAL)  # grounded answer
print(response)  # display answer with citations

The course CSBP 119, titled "Algorithms and Problem Solving," is offered by the UAEU Computer Science program. 

According to the course description [1 • description • CSBP_119::description], this course introduces students to problem-solving methods and program development, including the role of algorithms in the problem-solving process, implementation strategies for algorithms, and basic algorithms. Program design strategies are also covered, including implementation using a programming language that supports modular design and includes I/O, events, control structures, arrays, and functions.

The course is credited for 3 credits [2 • credits • CSBP_119::credits].
